# Proportional RAG – LLM generation (Colab)

Runs the pipeline end to end: **candidate pool (K=20) → selector (Top-k / MMR) → ALCE-style prompt → LLM → JSON output**.

Before running: `Runtime → Change runtime type → T4 GPU`. Then run the cells top to bottom.

Start with `N_QUESTIONS = 10` to check everything works, then increase to 50.

In [ ]:
# 1. Check the GPU
!nvidia-smi

In [ ]:
# 2. Install dependencies (transformers and torch are preinstalled on Colab)
!pip -q install -U sentence-transformers accelerate

In [ ]:
# 3. Get the shared candidate pools from the group repo
!git clone -q https://github.com/saminNazmus201/Project-In-Data-Science-4A.git repo

import glob
pool_files = glob.glob("repo/**/asqa_gtr_n50_k20.json", recursive=True)
print(pool_files)
POOL_PATH = pool_files[0]

In [ ]:
# 4. Settings – change these here only
N_QUESTIONS    = 10                                # start small, then 50
K_SELECT       = 5                                 # passages given to the LLM
LAM            = 0.5                               # MMR trade-off
METHODS        = ["topk", "mmr"]
EMBED_MODEL    = "BAAI/bge-base-en-v1.5"           # must match the one used locally / by Nazmus
LLM_MODEL      = "Qwen/Qwen2.5-3B-Instruct"        # placeholder until the group decides
MAX_NEW_TOKENS = 300

In [ ]:
# 5. Load the candidate pools
import json

with open(POOL_PATH) as f:
    pool = json.load(f)

data = pool[:N_QUESTIONS]
print(len(pool), "questions in file, using", len(data))
print("fields:", list(data[0].keys()))
print("doc fields:", list(data[0]["docs"][0].keys()))
print("docs per question:", len(data[0]["docs"]))

In [ ]:
# 6. Selectors: same interface as in the repo -> (question, docs, k) -> selected docs
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL, device="cuda")

def doc_text(d):
    return f"{d['title']} {d['text']}"

def select_topk(question, docs, k):
    # docs are already in retrieval order
    return docs[:k]

def select_mmr(question, docs, k, lam=LAM):
    q = embedder.encode(question, normalize_embeddings=True)
    p = embedder.encode([doc_text(d) for d in docs], normalize_embeddings=True)
    rel = p @ q          # relevance to the question
    sim = p @ p.T        # passage-passage similarity
    selected = [int(np.argmax(rel))]
    while len(selected) < k:
        best, best_score = None, -1e9
        for i in range(len(docs)):
            if i in selected:
                continue
            score = lam * rel[i] - (1 - lam) * max(sim[i][j] for j in selected)
            if score > best_score:
                best, best_score = i, score
        selected.append(best)
    return [docs[i] for i in selected]

SELECTORS = {"topk": select_topk, "mmr": select_mmr}

In [ ]:
# 7. ALCE-style prompt (numbered passages, answer with citations)
INSTRUCTION = (
    "Write an accurate, engaging, and concise answer for the given question using only "
    "the provided search results (some of which might be irrelevant) and cite them properly. "
    "Use an unbiased and journalistic tone. Always cite for any factual claim. "
    "When citing several search results, use [1][2][3]. Cite at least one document and at most "
    "three documents in each sentence. If multiple documents support the sentence, only cite a "
    "minimal set of documents."
)

def build_prompt(question, docs):
    doc_block = "\n\n".join(
        f"Document [{i+1}](Title: {d['title']}): {d['text']}" for i, d in enumerate(docs)
    )
    return f"{INSTRUCTION}\n\nQuestion: {question}\n\n{doc_block}\n\nAnswer:"

print(build_prompt(data[0]["question"], select_topk(data[0]["question"], data[0]["docs"], K_SELECT))[:1500])

In [ ]:
# 8. Load the LLM
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16, device_map="auto")
model.eval()

def generate(prompt):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# 9. Run generation for each method and save ALCE-style JSON ({"args": ..., "data": [...]})
import os, copy
from tqdm.auto import tqdm

os.makedirs("outputs", exist_ok=True)
model_tag = LLM_MODEL.split("/")[-1]
saved = []

for method in METHODS:
    select = SELECTORS[method]
    results = []
    for ex in tqdm(data, desc=method):
        selected = select(ex["question"], ex["docs"], K_SELECT)
        item = copy.deepcopy(ex)
        item["docs"] = selected                      # citations [1]..[k] refer to these
        item["method"] = method
        item["output"] = generate(build_prompt(ex["question"], selected))
        results.append(item)

    path = f"outputs/{method}_{model_tag}_n{N_QUESTIONS}_k{K_SELECT}.json"
    args = {"method": method, "model": LLM_MODEL, "embed_model": EMBED_MODEL,
            "k": K_SELECT, "lam": LAM if method == "mmr" else None, "n_questions": N_QUESTIONS}
    with open(path, "w") as f:
        json.dump({"args": args, "data": results}, f, indent=2)
    saved.append(path)
    print("saved", path)

In [ ]:
# 10. Quick sanity check: one example + simple str-EM (share of qa_pairs answered)
import re

def str_em(output, qa_pairs):
    out = output.lower()
    hits = [any(a.lower() in out for a in qa["short_answers"]) for qa in qa_pairs]
    return sum(hits) / len(hits)

for path in saved:
    with open(path) as f:
        res = json.load(f)["data"]
    em = np.mean([str_em(r["output"], r["qa_pairs"]) for r in res])
    cited = np.mean([bool(re.search(r"\[\d+\]", r["output"])) for r in res])
    print(f"{path}\n  str-EM: {em:.3f}   answers with citations: {cited:.0%}")

print("\nExample question:", res[0]["question"])
print("Output:", res[0]["output"])

In [ ]:
# 11. Download the output files (or copy them into the repo and push)
from google.colab import files
for path in saved:
    files.download(path)

**Notes**
- The str-EM above is a quick check only; the official numbers come from ALCE's `eval.py` (William's part).
- ALCE's citation evaluation uses a large NLI model (T5-XXL), which is too big for the free Colab GPU. Worth mentioning to William.
- To change the LLM later, only edit `LLM_MODEL` in cell 4.